Цель эксперимента: Построить классические Baseline-модели машинного обучения (Логистическую регрессию и Случайный лес) для фиксации базового уровня метрик качества.
Какие данные используются: Датасет cleaned_bnpl_data.csv с применением One-Hot Encoding (pd.get_dummies) для текстовых признаков и масштабированием (StandardScaler) числовых параметров под линейные алгоритмы.
Какие основные выводы: Модель LogisticRegression зафиксировала стартовую точку качества. Модель RandomForestClassifier показала более высокие результаты за счет учета нелинейных связей в данных, а полученные веса моделей были сохранены в artifacts/models/ для дальнейшего кросс-сравнения.

In [3]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import os
import joblib

from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
print('Working dir:', Path.cwd())

df = pd.read_csv('data/cleaned_bnpl_data.csv')
if 'Target' not in df.columns:
    if 'Repayment_Status' in df.columns:
        df['Target'] = (df['Repayment_Status'] == 'Defaulted').astype(int)
    else:
        raise KeyError('Target not found in data/cleaned_bnpl_data.csv')
X = df.drop(columns=['Target'])

y = df['Target']

X_dummy = pd.get_dummies(X, columns=['Gender', 'Purchase_Category', 'BNPL_Provider', 
                                     'Device_Type', 'Connection_Type', 'Browser'])
X_train_d, X_test_d, y_train, y_test = train_test_split(X_dummy, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_d)
X_test_scaled = scaler.transform(X_test_d)

# --- LOGISTIC REGRESSION ---
baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train_scaled, y_train)
base_preds = baseline.predict_proba(X_test_scaled)[:, 1]
# --- RANDOM FOREST ---
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_preds = rf_model.predict_proba(X_test_scaled)[:, 1]

os.makedirs('artifacts/models', exist_ok=True)
joblib.dump(baseline, 'artifacts/models/baseline_logreg.pkl')
joblib.dump(rf_model, 'artifacts/models/random_forest.pkl')


Working dir: c:\Users\fedor\Documents\proga\pet_scoring


['artifacts/models/random_forest.pkl']